In [1]:
import pandas as pd

df = pd.read_csv("../data/reviews.csv")

df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [2]:
df.shape

(568454, 10)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 568454 entries, 0 to 568453
Data columns (total 10 columns):
 #   Column                  Non-Null Count   Dtype
---  ------                  --------------   -----
 0   Id                      568454 non-null  int64
 1   ProductId               568454 non-null  str  
 2   UserId                  568454 non-null  str  
 3   ProfileName             568428 non-null  str  
 4   HelpfulnessNumerator    568454 non-null  int64
 5   HelpfulnessDenominator  568454 non-null  int64
 6   Score                   568454 non-null  int64
 7   Time                    568454 non-null  int64
 8   Summary                 568427 non-null  str  
 9   Text                    568454 non-null  str  
dtypes: int64(5), str(5)
memory usage: 43.4 MB


In [4]:
df.isnull().sum()

Id                         0
ProductId                  0
UserId                     0
ProfileName               26
HelpfulnessNumerator       0
HelpfulnessDenominator     0
Score                      0
Time                       0
Summary                   27
Text                       0
dtype: int64

In [5]:
df['Score'].value_counts().sort_index()

Score
1     52268
2     29769
3     42640
4     80655
5    363122
Name: count, dtype: int64

In [6]:
data = df[df['Score'] != 3]
# removing neutral reviews

In [7]:
data.shape

(525814, 10)

In [8]:
data['sentiment'] = data['Score'].apply(lambda x: 1 if x >= 4 else 0)

In [9]:
data[['Score', 'sentiment']].head()

,Score,sentiment
0,5,1
1,1,0
2,4,1
3,2,0
4,5,1


In [10]:
data['sentiment'].value_counts()
# imbalanced dataset , classes are imbalanced


sentiment
1    443777
0     82037
Name: count, dtype: int64

In [11]:
data = data.sample(50000, random_state=42)

In [12]:
data.shape

(50000, 11)

In [13]:
x = data['Text']
y = data['sentiment']

In [14]:
x.head()

443593    This is a very high quality dog food with meat...
136949    I love this cake mix and the other 3 mixes as ...
520459    A nice strong brew. I am new to Keurig and hav...
219515    I just found PB2 and PB2 with chocolate and I ...
471273    Delightful mint tea as one would expect. Note ...
Name: Text, dtype: str

In [15]:
y.head()

443593    1
136949    1
520459    1
219515    1
471273    1
Name: sentiment, dtype: int64

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state=42, stratify=y)
# stratify=y
# your complete dataset is roughly:
# 84% positive
# 16% negative
# Training → 84% positive / 16% negative
# Testing  → 84% positive / 16% negative

In [17]:
print(X_train.shape)
print(X_test.shape)

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))
# normalize=True
# converts those counts into proportions/percentages.

(40000,)
(10000,)
sentiment
1    0.84295
0    0.15705
Name: proportion, dtype: float64
sentiment
1    0.8429
0    0.1571
Name: proportion, dtype: float64


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizor = TfidfVectorizer()

X_train_tfidf = vectorizor.fit_transform(X_train)
X_test_tfidf = vectorizor.transform(X_test)

In [19]:
print(X_train_tfidf.shape)
print(X_test_tfidf.shape)
# 38,187 = number of unique words/features in our vocabulary.

(40000, 38187)
(10000, 38187)


In [20]:
list(vectorizor.vocabulary_.keys())[:20]

['this',
 'body',
 'wash',
 'lathers',
 'up',
 'well',
 'but',
 'you',
 'need',
 'to',
 'use',
 'good',
 'amount',
 'it',
 'isn',
 'concentrated',
 'the',
 'scent',
 'is',
 'similar']

In [21]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

In [22]:
model.fit(X_train_tfidf, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [23]:
y_pred = model.predict(X_test_tfidf)

In [24]:
print(y_pred[:20])

[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 0]


In [25]:
# score = w1&​x1 ​+ w2*​x2 ​+ w3*​x3 ​+...+ b
# x = tf-idf val, w = learned wt

# delicious:
# TF-IDF = 0.71
# weight  = +2.5

# terrible:
# TF-IDF = 0
# weight  = -2.7

In [26]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.9164


In [27]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix (y_test, y_pred)

print(cm)

[[ 864  707]
 [ 129 8300]]


In [28]:
#                   PREDICTED
#                 Negative Positive

# ACTUAL Negative    TN       FP
# ACTUAL Positive    FN       TP

In [29]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.87      0.55      0.67      1571
           1       0.92      0.98      0.95      8429

    accuracy                           0.92     10000
   macro avg       0.90      0.77      0.81     10000
weighted avg       0.91      0.92      0.91     10000



In [30]:
# Precision = 87%       *** GOOD ****
# Precision -if model predicts 100 neg values -> 87 of them are correct
#  When your model says: "This review is Negative."
# it's correct about 87% of the time. 

# Recall = 864 / (707 + 864) = 0.55   **** POOR *****
# Recall - if there are total 100 neg reviews -> model pred 55
# Your model detects only about 55% of all the negative reviews.



In [31]:
import joblib

joblib.dump(model, "../models/sentiment_model.pkl")
joblib.dump(vectorizor, "../models/tfidf_vectorizer.pkl")

['../models/tfidf_vectorizer.pkl']

In [32]:
loaded_model = joblib.load('../models/sentiment_model.pkl')
loaded_vectorizer = joblib.load('../models/tfidf_vectorizer.pkl')

In [33]:
review = [ "Amazing product, I will definitely buy it again"]

review_tfidf = loaded_vectorizer.transform(review)
prediction = loaded_model.predict(review_tfidf)
print(prediction)

[1]


#Experiment 1: class imbalance.

TF-IDF
   ->
Logistic Regression
   ->
class_weight="balanced"

Our training data looks roughly like:
Positive → 84%
Negative → 16%

Now using class_wt = 'balanced'
Positive example → lower weight
Negative example → higher weight

In [34]:
balanced_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

balanced_model.fit(X_train_tfidf, y_train)
balanced_pred = balanced_model.predict(X_test_tfidf)

In [35]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print('Accuracy :', accuracy_score(y_test, balanced_pred))
print()
print('Classification Report:')
print(classification_report(y_test, balanced_pred))
print()
print('Confusion Matrix:')
print(confusion_matrix(y_test, balanced_pred))

Accuracy : 0.8954

Classification Report:
              precision    recall  f1-score   support

           0       0.62      0.86      0.72      1571
           1       0.97      0.90      0.94      8429

    accuracy                           0.90     10000
   macro avg       0.80      0.88      0.83     10000
weighted avg       0.92      0.90      0.90     10000


Confusion Matrix:
[[1347  224]
 [ 822 7607]]


In [36]:
review = [ "the food was good "]

review_tfidf = loaded_vectorizer.transform(review)
prediction = balanced_model.predict(review_tfidf)
print(prediction)

[1]


#Experiment 2: changing the decision threshold.#

So increasing the threshold means:

The model becomes more conservative about calling something Positive.

Higher threshold
      ->
More Negative predictions
      ->
Usually catches more actual Negatives
      ->
Negative recall tends to increase

In [37]:
model.predict_proba(X_test_tfidf)

array([[0.23786759, 0.76213241],
       [0.01212972, 0.98787028],
       [0.01296048, 0.98703952],
       ...,
       [0.00115469, 0.99884531],
       [0.02571169, 0.97428831],
       [0.104155  , 0.895845  ]], shape=(10000, 2))

In [38]:
y_prob = model.predict_proba(X_test_tfidf)[:, 1]

In [39]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

thresholds = [0.3, 0.4, 0.5, 0.6, 0.65, 0.7]

for threshold in thresholds :
    y_pred_threshold = (y_prob >= threshold).astype(int)
    
    print("Threshold :", threshold)
    print("Accuracy:", accuracy_score(y_test, y_pred_threshold))
    print("Negative Precision:", precision_score(y_test, y_pred_threshold, pos_label=0))
    print("Negative Recall:", recall_score(y_test, y_pred_threshold, pos_label=0))
    print("Negative F1:", f1_score(y_test, y_pred_threshold, pos_label=0))
    print()

Threshold : 0.3
Accuracy: 0.89
Negative Precision: 0.9337016574585635
Negative Recall: 0.3227243793761935
Negative F1: 0.4796594134342479

Threshold : 0.4
Accuracy: 0.9035
Negative Precision: 0.899736147757256
Negative Recall: 0.4341183959261617
Negative F1: 0.5856590811507084

Threshold : 0.5
Accuracy: 0.9164
Negative Precision: 0.8700906344410876
Negative Recall: 0.5499681731381286
Negative F1: 0.6739469578783152

Threshold : 0.6
Accuracy: 0.9225
Negative Precision: 0.817384370015949
Negative Recall: 0.6524506683640993
Negative F1: 0.7256637168141593

Threshold : 0.65
Accuracy: 0.9224
Negative Precision: 0.7762334954829743
Negative Recall: 0.7110120942075111
Negative F1: 0.7421926910299004

Threshold : 0.7
Accuracy: 0.9165
Negative Precision: 0.7214199759326113
Negative Recall: 0.7632081476766391
Negative F1: 0.7417259511289823



EXPERIMENT 3 : Linear SVM

In [40]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC()

svm_model.fit (X_train_tfidf, y_train)
svm_pred = svm_model.predict (X_test_tfidf)

In [41]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy :", accuracy_score(y_test, svm_pred))
print()
print("Classification Report :\n", classification_report(y_test, svm_pred))
print()
print("Confusion Matrix :", confusion_matrix(y_test, svm_pred))

Accuracy : 0.9265

Classification Report :
               precision    recall  f1-score   support

           0       0.82      0.68      0.75      1571
           1       0.94      0.97      0.96      8429

    accuracy                           0.93     10000
   macro avg       0.88      0.83      0.85     10000
weighted avg       0.92      0.93      0.92     10000


Confusion Matrix : [[1075  496]
 [ 239 8190]]


In [42]:
review = [ "the food was not bad "]

review_tfidf = loaded_vectorizer.transform(review)
prediction = svm_model.predict(review_tfidf)
print(prediction)

[0]


Experiment 4 — Unigrams + Bigrams

In [43]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_bigram = TfidfVectorizer(
    ngram_range=(1,2)
)

X_train_bigram = tfidf_bigram.fit_transform(X_train)
X_test_bigram = tfidf_bigram.transform(X_test)

print(X_train_bigram.shape)
print(X_test_bigram.shape)

(40000, 659163)
(10000, 659163)


Training bigram model on SVC

In [44]:
svm_bigram = LinearSVC()

svm_bigram.fit(X_train_bigram, y_train)
svm_bigram_pred = svm_bigram.predict(X_test_bigram)

In [45]:
print("Accuracy:", accuracy_score(y_test, svm_bigram_pred))

print("\nClassification Report:")
print(classification_report(y_test, svm_bigram_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, svm_bigram_pred))

Accuracy: 0.9427

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.72      0.80      1571
           1       0.95      0.98      0.97      8429

    accuracy                           0.94     10000
   macro avg       0.92      0.85      0.88     10000
weighted avg       0.94      0.94      0.94     10000


Confusion Matrix:
[[1132  439]
 [ 134 8295]]


In [46]:
review = [ "the food was not terrible"]

review_tfidf = tfidf_bigram.transform(review)
prediction = svm_bigram.predict(review_tfidf)
print(prediction)

[0]


In [47]:
"not terrible" in tfidf_bigram.vocabulary_

True

In [48]:
score = svm_bigram.decision_function(review_tfidf)

print(score)

[-1.60954491]


In [49]:
import numpy as np
feature_names = tfidf_bigram.get_feature_names_out()
weights = svm_bigram.coef_[0]

for word in ["terrible", "not", "not terrible"]:
    idx = np.where(feature_names == word)[0]

    if len(idx) > 0:
        print(word, weights[idx[0]])
    else:
        print(word, "NOT FOUND")

terrible -3.0856295691527826
not -3.6509186787973107
not terrible 0.041473284571896946


In [50]:
import joblib
joblib.dump(svm_bigram, '../models/sentiment_svm.pkl')
joblib.dump(tfidf_bigram, '../models/tfidf_bigram.pkl')

['../models/tfidf_bigram.pkl']

In [51]:
loaded_svm = joblib.load('../models/sentiment_svm.pkl')
loaded_tfidf = joblib.load('../models/tfidf_bigram.pkl')

In [52]:
review = ["The food was amazing and delicious"]

review_tfidf = loaded_tfidf.transform(review)

prediction = loaded_svm.predict(review_tfidf)

print(prediction)

[1]


MODEL COMPARISON

In [53]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

In [54]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Linear SVM": LinearSVC(),
    "Naive Bayes": MultinomialNB()
}

In [55]:
for name, model in models.items() :
    model.fit(X_train_bigram, y_train)
    y_pred = model.predict(X_test_bigram)
    accuracy = accuracy_score(y_test, y_pred)
    print(name, ":", accuracy)
    

Logistic Regression : 0.9147
Linear SVM : 0.9427
Naive Bayes : 0.8429


In [56]:
from sklearn.metrics import classification_report

for name, model in models.items():
    model.fit(X_train_bigram, y_train)
    y_pred = model.predict(X_test_bigram)
    
    print("\n","=" * 50)
    print(name)
    print("=" * 50)
    print(classification_report(y_test, y_pred))


Logistic Regression
              precision    recall  f1-score   support

           0       0.92      0.50      0.65      1571
           1       0.91      0.99      0.95      8429

    accuracy                           0.91     10000
   macro avg       0.92      0.75      0.80     10000
weighted avg       0.92      0.91      0.90     10000


Linear SVM
              precision    recall  f1-score   support

           0       0.89      0.72      0.80      1571
           1       0.95      0.98      0.97      8429

    accuracy                           0.94     10000
   macro avg       0.92      0.85      0.88     10000
weighted avg       0.94      0.94      0.94     10000


Naive Bayes
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      1571
           1       0.84      1.00      0.91      8429

    accuracy                           0.84     10000
   macro avg       0.42      0.50      0.46     10000
weighted avg       0.71     

d:\Desktop\food review analyzer\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Desktop\food review analyzer\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\Desktop\food review analyzer\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.sh

Find what the SVM gets wrong

In [57]:
svm = LinearSVC()

svm.fit(X_train_bigram, y_train)

y_pred = svm.predict(X_test_bigram)

In [58]:
wrong = y_pred != y_test

wrong_reviews = X_test[wrong]
wrong_actual = y_test[wrong]
wrong_predicted = y_pred[wrong]

In [59]:
errors = pd.DataFrame({
    "Review" : wrong_reviews,
    "Actual": wrong_actual,
    "Predicted" : wrong_predicted
})

errors.head(20)

,Review,Actual,Predicted
291222,Avoid peanuts if you have high cholesterol. Y...,0,1
132475,These things taste like jumping off a diving b...,0,1
37275,I didn't really care for this tea that much. I...,0,1
195384,The bars were too sweet for me and kind of dry...,0,1
203797,"As a happy consumer of Earth Grains Faro, I wa...",0,1
109075,This kit saves you the time of making your own...,0,1
201346,Received the wrong item.......I received ORGAN...,0,1
327920,I've taken a handful of pre-workout supplement...,0,1
385133,I'm always looking for a single-use coffee tha...,0,1
290278,I have been using the full line of Lipton's Gr...,0,1


In [60]:
pd.set_option("display.max_colwidth", None)

errors.head(20)

,Review,Actual,Predicted
291222,"Avoid peanuts if you have high cholesterol. Your belly will expand and UNDIGESTED PEANUTS, partialy whole, will appear in your<br />EXCRETIONS. Make your own peanut butter with this product.",0,1
132475,These things taste like jumping off a diving board feet first into an over-chlorinated swimming pool. I did give this product an extra star though for the entertainment value of tricking all my coworkers one by one into trying one and observing the horror they experience. 90% can't even get it down without spitting it back out.,0,1
37275,"I didn't really care for this tea that much. I've been trying out teas to find which ones I like, I've tried about 30 so far. I like Celestial seasoning's Mandarin Orchard green tea and Salada green tea better than this. I find most Stash brands to be astringent tasting, leaving my mouth feeling dry. If I really watch how hot the water is and steep for only a short time, then the taste is bearable but very weak. My husband also doesn't like any of the stash teas that we've tried.",0,1
195384,The bars were too sweet for me and kind of dry. The service was wonderful.,0,1
203797,"As a happy consumer of Earth Grains Faro, I was disappointed to find Amazon out of stock--and ordered the Roland brand instead. Where the Earth Grains brand is tender, with separate grains, good texture and a mild, wheaty flavour-the Roland brand is tough, gritty and altogether unpleasant.",0,1
109075,"This kit saves you the time of making your own, but at three times the cost. Anyone can buy bulk planting sponges or rock wool and make their own for a couple dollars. However, if you don't care about the price and want to make Aero Garden planting fast, easy, and thoughtless, then this is made for you. The kit is great, just uncomfortably priced for what you get.",0,1
201346,Received the wrong item.......I received ORGANIC RAW CACAO POWDER which still contains the fat. I didn't notice until after I had opened and used the product.<br />Not too happy about that!,0,1
327920,"I've taken a handful of pre-workout supplements (it's my favorite category of fitness supplements), such as N.O. Xplode, NanoVapor, Jack3d, and 1MR, and this is the first time I've felt compelled to write a review about one, because this one is just really bad, in my experience. With even just having half the recommended dose (on an empty stomach, with proper pre-workout nutrition and hydration...I know the PWO drill), I'm already swearing off this stuff. It does nothing but make me feel sick to my stomach, and all I want to do is just drink water and NOT work out, lest I actually vomit while lifting.<br /><br />I'm dumbfounded by some of the overly positive reviews out there, but hey, to each his own, but this supplement is no good.",0,1
385133,"I'm always looking for a single-use coffee that is strong enough for my taste without using two pods ... this isn't it. In fact, I've had cheap instant coffee that taste better.<br /><br />I use my pod for a quick cup or take them with me when I'm traveling, so I don't expect the product to taste like my fresh ground brewed coffee. But I do expect flavor. The flavor of this coffee was similar to re-used grounds.<br /><br />The only redeeming quality about this Marley product is that it is a free-trade, organic product.<br /><br />I'm not sure what I'm going to do with the left-over packages because I don't want to insult guests by serving it. It seems a waste to just throw it away. Maybe I can use it in the garden.",0,1
290278,"I have been using the full line of Lipton's Green Tea to Go for a couple of years now, and just love it. Have also turned countless friends onto it as well. Love the benefits of green tea, being able to avoid soda...and that it is sweetened with Splenda. Splenda was the primary reason for chosing the Lipton line. But NOW all that has changed and Lipton has made the decision to switch to Aspartame...which doctors tell you to avoid if you experience headach

In [61]:
pd.crosstab(
    errors["Actual"],
    errors['Predicted']
)

Predicted,0,1
Actual,,
0,0,439
1,134,0


In [62]:
decision_scores = svm.decision_function(X_test_bigram)

errors['Score'] = decision_scores[wrong]

errors['AbsScore'] = errors['Score'].abs()

errors.sort_values('AbsScore').head(20)

,Review,Actual,Predicted,Score,AbsScore
260743,I'm very dissappointed that the regular flavors of these soda syrups contain artificial sweeteners. Not all of the regular flavors are available in the Natural flavors so you tend to miss out on many of the varieties. Too bad. I might have to look into making my own syrups.,0,1,0.000470,0.000470
472319,I love these but that is a ridiculous price...Costco used to sell same thing for $9.95..and still had to break even at least...,1,0,-0.001624,0.001624
319153,"So you want some tasty flavorful potato chips, seriously why bother with these. Please understand they are baked. Even so, they taste like seasoned cardboard. Ur better off with a rice cake. Or how about you walk around the block and then enjoy a handful of the standard kettle chips, well worth it!<br /><br />This product Skip!",0,1,0.002801,0.002801
536617,"I thought these would be healthy, but I was wrong. I tried one and it was so sweet it hurt my teeth (it has 14 gr of sugar for each small square). In addition, it aggravated my heartburn. If you are looking for a healthy snack, look elsewhere... these are like candy...",0,1,0.003099,0.003099
87073,"I did not post this under packaging review, because the order arrived fully sealed, no markings or damage to the outside box, but 8 out of the 12 cans were severly dented, so I had to assume they had been packaged that way. They were even hard to open with the can opener. I will still get my subscribed monthly order, but if it happens again I will probably cancel it. I have no problems with the quality of the product, just very dissatisfied with the way it was mailed.",0,1,0.004946,0.004946
509429,"I did not post this under packaging review, because the order arrived fully sealed, no markings or damage to the outside box, but 8 out of the 12 cans were severly dented, so I had to assume they had been packaged that way. They were even hard to open with the can opener. I will still get my subscribed monthly order, but if it happens again I will probably cancel it. I have no problems with the quality of the product, just very dissatisfied with the way it was mailed.",0,1,0.004946,0.004946
47379,"I came back to make a purchase only to find that the price has *doubled* in the course of five months.<br />I paid $27.42 for this item from Amazon in November, 2010. I'll have to take a pass, as this price is too rich for my blood.<br /><br />Edit: I see now the price has dropped again! So my review doesn't make sense any longer. Trust me, when I wrote this, the 2-pack was $50-something.<br /><br />I upped the stars because I found the cashews to be good.",1,0,-0.005263,0.005263
201346,Received the wrong item.......I received ORGANIC RAW CACAO POWDER which still contains the fat. I didn't notice until after I had opened and used the product.<br />Not too happy about that!,0,1,0.006589,0.006589
317899,Just got a Keurig and wanted to try out a non coffee alternative. I've always been a big apple fan and these did not let me down. Some instant ciders can taste very fake but not these. Will buy again.,1,0,-0.007201,0.007201
232939,"BPA is not proven to harm children or adults, however studies have led federal health officials to express concern about the safety of BPA. Eden Foods has been using BPA free cans since 1999. Natural Value told me their cans are BPA lined and they are considering a change in the future.<br /><br />The beans are good, firm and a bit too salty. I will not buy any Natural Value product because of the BPA issue. All their cans are BPA lined. I feel cheated by a company that chooses to market an ORGANIC product in a can with a BPA lining when other choices have been available for over 10 years.<br /><br />I'll buy Eden Foods canned products. All canned tomatoes are in BPA lined cans per government regulations. Freeze your own or buy in glass jars or cartons.",0,1,0.007224,0.007224


In [63]:
errors.sort_values("AbsScore").head(20)[
    ["Review", "Actual", "Predicted", "Score", "AbsScore"]
]

,Review,Actual,Predicted,Score,AbsScore
260743,I'm very dissappointed that the regular flavors of these soda syrups contain artificial sweeteners. Not all of the regular flavors are available in the Natural flavors so you tend to miss out on many of the varieties. Too bad. I might have to look into making my own syrups.,0,1,0.000470,0.000470
472319,I love these but that is a ridiculous price...Costco used to sell same thing for $9.95..and still had to break even at least...,1,0,-0.001624,0.001624
319153,"So you want some tasty flavorful potato chips, seriously why bother with these. Please understand they are baked. Even so, they taste like seasoned cardboard. Ur better off with a rice cake. Or how about you walk around the block and then enjoy a handful of the standard kettle chips, well worth it!<br /><br />This product Skip!",0,1,0.002801,0.002801
536617,"I thought these would be healthy, but I was wrong. I tried one and it was so sweet it hurt my teeth (it has 14 gr of sugar for each small square). In addition, it aggravated my heartburn. If you are looking for a healthy snack, look elsewhere... these are like candy...",0,1,0.003099,0.003099
87073,"I did not post this under packaging review, because the order arrived fully sealed, no markings or damage to the outside box, but 8 out of the 12 cans were severly dented, so I had to assume they had been packaged that way. They were even hard to open with the can opener. I will still get my subscribed monthly order, but if it happens again I will probably cancel it. I have no problems with the quality of the product, just very dissatisfied with the way it was mailed.",0,1,0.004946,0.004946
509429,"I did not post this under packaging review, because the order arrived fully sealed, no markings or damage to the outside box, but 8 out of the 12 cans were severly dented, so I had to assume they had been packaged that way. They were even hard to open with the can opener. I will still get my subscribed monthly order, but if it happens again I will probably cancel it. I have no problems with the quality of the product, just very dissatisfied with the way it was mailed.",0,1,0.004946,0.004946
47379,"I came back to make a purchase only to find that the price has *doubled* in the course of five months.<br />I paid $27.42 for this item from Amazon in November, 2010. I'll have to take a pass, as this price is too rich for my blood.<br /><br />Edit: I see now the price has dropped again! So my review doesn't make sense any longer. Trust me, when I wrote this, the 2-pack was $50-something.<br /><br />I upped the stars because I found the cashews to be good.",1,0,-0.005263,0.005263
201346,Received the wrong item.......I received ORGANIC RAW CACAO POWDER which still contains the fat. I didn't notice until after I had opened and used the product.<br />Not too happy about that!,0,1,0.006589,0.006589
317899,Just got a Keurig and wanted to try out a non coffee alternative. I've always been a big apple fan and these did not let me down. Some instant ciders can taste very fake but not these. Will buy again.,1,0,-0.007201,0.007201
232939,"BPA is not proven to harm children or adults, however studies have led federal health officials to express concern about the safety of BPA. Eden Foods has been using BPA free cans since 1999. Natural Value told me their cans are BPA lined and they are considering a change in the future.<br /><br />The beans are good, firm and a bit too salty. I will not buy any Natural Value product because of the BPA issue. All their cans are BPA lined. I feel cheated by a company that chooses to market an ORGANIC product in a can with a BPA lining when other choices have been available for over 10 years.<br /><br />I'll buy Eden Foods canned products. All canned tomatoes are in BPA lined cans per government regulations. Freeze your own or buy in glass jars or cartons.",0,1,0.007224,0.007224


In [64]:
duplicate_count = df['Text'].duplicated().sum()

print("Duplicate reviews:", duplicate_count)
print("Percentage:", duplicate_count/len(df) * 100)

Duplicate reviews: 174875
Percentage: 30.763263166412763


In [65]:
duplicate_texts = df[df["Text"].duplicated(keep=False)]

conflicts = duplicate_texts.groupby("Text")["Score"].nunique()

print("Duplicate texts:", duplicate_texts["Text"].nunique())
print("Duplicate texts with conflicting labels:", (conflicts > 1).sum())

Duplicate texts: 58040
Duplicate texts with conflicting labels: 95


USING class_weight="balanced"

Class 0 → 1,571
Class 1 → 8,429

Class 0 mistake → HIGHER penalty
Class 1 mistake → LOWER relative penalty

In [66]:
svm_balanced = LinearSVC(class_weight='balanced')

svm_balanced.fit(X_train_bigram, y_train)

y_pred_balanced = svm_balanced.predict(X_test_bigram)

In [67]:
print("Classification Report:")
print(classification_report(y_test, y_pred_balanced))

print("Accuracy :", accuracy_score(y_test, y_pred_balanced))

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.80      0.82      1571
           1       0.96      0.97      0.97      8429

    accuracy                           0.94     10000
   macro avg       0.90      0.89      0.89     10000
weighted avg       0.94      0.94      0.94     10000

Accuracy : 0.9446


Think of C as controlling how strongly the SVM cares about classification mistakes on the training data.

In [68]:
from sklearn.model_selection import GridSearchCV

svm = LinearSVC(class_weight='balanced')

param_grid = {
    'C' : [0.1, 0.5, 1, 2, 5]
}

grid = GridSearchCV (
    svm,
    param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

grid.fit(X_train_bigram, y_train)

print("Best C:", grid.best_params_)
print("Best CV score:", grid.best_score_)

Best C: {'C': 0.5}
Best CV score: 0.8925708348617633


In [69]:
best_svm = grid.best_estimator_

In [70]:
y_pred_final = best_svm.predict(X_test_bigram)

In [71]:
print(classification_report(y_test, y_pred_final))

              precision    recall  f1-score   support

           0       0.82      0.83      0.82      1571
           1       0.97      0.97      0.97      8429

    accuracy                           0.94     10000
   macro avg       0.89      0.90      0.89     10000
weighted avg       0.94      0.94      0.94     10000



In [72]:
print('Accuracy:', accuracy_score(y_test, y_pred_final))

Accuracy: 0.9435


In [73]:
results = grid.cv_results_

for c, mean, std in zip (
    results['param_C'],
    results['mean_test_score'],
    results['std_test_score']
) :
    print(
        "C:", c,
        'Mean F1:', round(mean, 4),
        "Std Dev:", round(std, 4) 
    )

C: 0.1 Mean F1: 0.8681 Std Dev: 0.0029
C: 0.5 Mean F1: 0.8926 Std Dev: 0.004
C: 1.0 Mean F1: 0.8919 Std Dev: 0.0038
C: 2.0 Mean F1: 0.8883 Std Dev: 0.0045
C: 5.0 Mean F1: 0.8849 Std Dev: 0.0042


Now We will try unigrams, bigrams and trigrams


In [74]:
tfidf_uni = TfidfVectorizer(
    ngram_range=(1,1)
)

X_train_uni = tfidf_uni.fit_transform(X_train)
X_test_uni = tfidf_uni.transform(X_test)

print("Unigram Features:" , X_train_uni.shape[1])

Unigram Features: 38187


In [75]:
tfidf_bi = TfidfVectorizer(
    ngram_range=(1,2)
)

X_train_bi = tfidf_bi.fit_transform(X_train)
X_test_bi = tfidf_bi.transform(X_test)

print("Bigram features:", X_train_bi.shape[1])


Bigram features: 659163


In [76]:
tfidf_tri = TfidfVectorizer(
    ngram_range=(1,3)
)

X_train_tri = tfidf_tri.fit_transform(X_train)
X_test_tri = tfidf_tri.transform(X_test)

print("Trigram features:", X_train_tri.shape[1])

Trigram features: 2347149


In [77]:
svm_uni = LinearSVC(
    C=0.5,
    class_weight='balanced'
)

svm_uni.fit(X_train_uni, y_train)
pred_uni = svm_uni.predict(X_test_uni)

print("Unigram accuracy:", accuracy_score(y_test, pred_uni))

print(classification_report(y_test, pred_uni))

Unigram accuracy: 0.9064
              precision    recall  f1-score   support

           0       0.66      0.83      0.74      1571
           1       0.97      0.92      0.94      8429

    accuracy                           0.91     10000
   macro avg       0.81      0.87      0.84     10000
weighted avg       0.92      0.91      0.91     10000



In [78]:
svm_bi = LinearSVC(
    C=0.5,
    class_weight='balanced'
)

svm_bi.fit(X_train_bi, y_train)
pred_bi = svm_bi.predict(X_test_bi)

print("Bigram accuracy:", accuracy_score(y_test, pred_bi))

print(classification_report(y_test, pred_bi))

Bigram accuracy: 0.9435
              precision    recall  f1-score   support

           0       0.82      0.83      0.82      1571
           1       0.97      0.97      0.97      8429

    accuracy                           0.94     10000
   macro avg       0.89      0.90      0.89     10000
weighted avg       0.94      0.94      0.94     10000



In [79]:
svm_tri = LinearSVC(
    C=0.5,
    class_weight='balanced'
)

svm_tri.fit(X_train_tri, y_train)
pred_tri = svm_tri.predict(X_test_tri)

print("Trigram accuracy:", accuracy_score(y_test, pred_tri))

print(classification_report(y_test, pred_tri))

Trigram accuracy: 0.9429
              precision    recall  f1-score   support

           0       0.83      0.80      0.82      1571
           1       0.96      0.97      0.97      8429

    accuracy                           0.94     10000
   macro avg       0.90      0.89      0.89     10000
weighted avg       0.94      0.94      0.94     10000



In [80]:
print(df['Text'].str.contains('<br', regex=False).sum())

142924


In [81]:
df.loc[df['Text'].str.contains('<br', regex=False), "Text"].iloc[0]

"I don't know if it's the cactus or the tequila or just the unique combination of ingredients, but the flavour of this hot sauce makes it one of a kind!  We picked up a bottle once on a trip we were on and brought it back home with us and were totally blown away!  When we realized that we simply couldn't find it anywhere in our city we were bummed.<br /><br />Now, because of the magic of the internet, we have a case of the sauce and are ecstatic because of it.<br /><br />If you love hot sauce..I mean really love hot sauce, but don't want a sauce that tastelessly burns your throat, grab a bottle of Tequila Picante Gourmet de Inclan.  Just realize that once you taste it, you will never want to use any other sauce.<br /><br />Thank you for the personal, incredible service!"

In [82]:
df['CleanText'] = df['Text'].str.replace("<br />", " ", regex=False)

In [83]:
print(df['Text'].iloc[0])
print(df['CleanText'].iloc[0])

I have bought several of the Vitality canned dog food products and have found them all to be of good quality. The product looks more like a stew than a processed meat and it smells better. My Labrador is finicky and she appreciates this product better than  most.
I have bought several of the Vitality canned dog food products and have found them all to be of good quality. The product looks more like a stew than a processed meat and it smells better. My Labrador is finicky and she appreciates this product better than  most.


In [84]:
print(df['CleanText'].str.contains("<br />", regex=False).sum())

0


In [85]:
print("Original:")
print(df["Text"].iloc[0])

print("\nCleaned:")
print(df["CleanText"].iloc[0])

Original:
I have bought several of the Vitality canned dog food products and have found them all to be of good quality. The product looks more like a stew than a processed meat and it smells better. My Labrador is finicky and she appreciates this product better than  most.

Cleaned:
I have bought several of the Vitality canned dog food products and have found them all to be of good quality. The product looks more like a stew than a processed meat and it smells better. My Labrador is finicky and she appreciates this product better than  most.


In [86]:
data["CleanText"] = data["Text"].str.replace(
    "<br />", " ", regex=False
)


In [87]:
X_train_clean = data.loc[X_train.index, 'CleanText']
X_test_clean = data.loc[X_test.index, 'CleanText']

In [88]:
tfidf_clean = TfidfVectorizer(
    ngram_range=(1,2)
)

X_train_clean_vec = tfidf_clean.fit_transform(X_train_clean)
X_test_clean_vec = tfidf_clean.transform(X_test_clean)

In [89]:
svm_clean = LinearSVC(
    C=0.5,
    class_weight='balanced'
)

svm_clean.fit(X_train_clean_vec, y_train)

pred_clean = svm_clean.predict(X_test_clean_vec)

In [90]:
print("Cleaned Accuracy:",
      accuracy_score(y_test, pred_clean))

print(classification_report(y_test, pred_clean))

Cleaned Accuracy: 0.9436
              precision    recall  f1-score   support

           0       0.81      0.83      0.82      1571
           1       0.97      0.96      0.97      8429

    accuracy                           0.94     10000
   macro avg       0.89      0.90      0.89     10000
weighted avg       0.94      0.94      0.94     10000



Negation Handling


In [91]:
negation_words = [
    "not",
    "no",
    "never",
    "don't",
    "didn't",
    "doesn't",
    "wasn't",
    "weren't",
    "can't",
    "couldn't",
    "won't",
    "wouldn't"
]

for word in negation_words:
    count = data['Text'].str.lower().str.contains(
        rf"\b{word}\b",
        regex=True,
        na=False
    ).sum()
    
    print(word, count)

not 18100
no 7467
never 3236
don't 6383
didn't 2507
doesn't 2468
wasn't 996
weren't 190
can't 3123
couldn't 968
won't 1467
wouldn't 862


In [97]:
examples = data[data['Text'].str.lower().str.contains(r"\bnot\b", regex = True,na = False)]['Text'].head(10)

for review in examples :
    print(review)
    print('-' * 80)

A nice strong brew. I am new to Keurig and have lived on French press for years. This coffee is strong enough to compete.  It is a little on the bitter side like some of Starbucks coffees but I consider that a plus. I would rate it 5 stars but it's not French press
--------------------------------------------------------------------------------
I just found PB2 and PB2 with chocolate and I am thrilled.  I love sugary goodies but I'm not a fan of calories and I don't like chocolate.  I thought I'd try PB2 with chocolate anyway and I'm so glad I did!  Eating a serving or two of PB2 with chocolate satisfies my sweet craving without breaking my calorie bank!  Sometimes, when I want to splurge, I spread some PB2 with chocolate on a graham cracker, spread some marshmallow fluff on another and sandwich them together! YUM!
--------------------------------------------------------------------------------
Delightful mint tea as one would expect. Note that each tea bag is NOT individually wrapped;

In [98]:
negations = {
    "not",
    "no",
    "never",
    "don't",
    "didn't",
    "doesn't",
    "wasn't",
    "weren't",
    "can't",
    "couldn't",
    "won't",
    "wouldn't"
}

In [101]:
def mark_negation (text) :
    words = text.lower().split()
    
    result = []
    
    negate_next = False
    
    for word in words :
        if word in negations :
            result.append(word)
            negate_next = True
            
        elif negate_next  :
            result.append(word + "_NEG")
            negate_next = False
            
        else :
            result.append(word)
            
    return " ".join(result)

In [102]:
print(mark_negation("The food was not terrible"))

print(mark_negation("I was not disappointed"))

print(mark_negation("I don't like this food"))

print(mark_negation("This product is very good"))

the food was not terrible_NEG
i was not disappointed_NEG
i don't like_NEG this food
this product is very good


In [103]:
data['NegationText'] = data['CleanText'].apply (mark_negation)

In [107]:
print(data[['CleanText', 'NegationText']].head(10).to_string())

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   CleanText                                                                                                                                                                                                                                                                                                                                                                                                            

In [108]:
X_train_neg = data.loc[X_train.index, "NegationText"]
X_test_neg = data.loc[X_test.index, "NegationText"]

In [109]:
tfidf_neg = TfidfVectorizer (
    ngram_range= (1,2)
)

X_train_neg_vec = tfidf_neg.fit_transform (X_train_neg)
X_test_neg_vec = tfidf_neg.transform (X_test_neg)

In [110]:
svm_neg = LinearSVC (
    C=0.5,
    class_weight='balanced'
)

svm_neg.fit (X_train_neg_vec, y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",0.5
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",'balanced'
,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseud

In [111]:
pred_neg = svm_neg.predict (X_test_neg_vec)

print ("Negation Accuracy: ", accuracy_score (y_test, pred_neg))
print (classification_report (y_test, pred_neg))

Negation Accuracy:  0.944
              precision    recall  f1-score   support

           0       0.82      0.83      0.82      1571
           1       0.97      0.97      0.97      8429

    accuracy                           0.94     10000
   macro avg       0.89      0.90      0.89     10000
weighted avg       0.94      0.94      0.94     10000



In [116]:
review = ["The food was not terrible."]

review_tfidf = tfidf_neg.transform(review)

prediction = svm_neg.predict(review_tfidf)

print(prediction)

[0]


Threshold Handling


In [112]:
decision_scores = svm_clean.decision_function (X_test_clean_vec)
print (decision_scores[:10])

[ 0.16141013  1.30901971  1.46842883  1.11345783  0.30684094  0.9668766
  1.22161935 -0.41329692  0.18475221  0.37518735]


In [113]:
thresholds = [-0.5, -0.25, 0, 0.25, 0.5, 0.75, 1.0]

for threshold in thresholds :
    pred = (decision_scores >= threshold).astype(int)
    
    print (
        "Threshold:", threshold,
        "Macro F1:", f1_score(y_test, pred, average="macro"),
        "Accuracy:", accuracy_score(y_test, pred)
    )

Threshold: -0.5 Macro F1: 0.8291457130329569 Accuracy: 0.9252
Threshold: -0.25 Macro F1: 0.8746048258426353 Accuracy: 0.9397
Threshold: 0 Macro F1: 0.8943393149770499 Accuracy: 0.9436
Threshold: 0.25 Macro F1: 0.8630062735204199 Accuracy: 0.917
Threshold: 0.5 Macro F1: 0.7910968389691646 Accuracy: 0.8539
Threshold: 0.75 Macro F1: 0.6764082648327413 Accuracy: 0.7345
Threshold: 1.0 Macro F1: 0.5402751485912465 Accuracy: 0.5709


In [114]:
import joblib

joblib.dump (
    tfidf_clean,
    '../models/tfidf_vectorizer_clean.pkl'
)

joblib.dump (
    svm_clean,
    '../models/sentiment_svm_clean.pkl'
)

['../models/sentiment_svm_clean.pkl']

In [115]:
import os

print(os.path.exists("../models/tfidf_vectorizer_clean.pkl"))
print(os.path.exists("../models/sentiment_svm_clean.pkl"))


True
True
